In [47]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import operator

load_dotenv()

chat_model = ChatOpenAI(model='gpt-4o-mini')

In [48]:
# We need structured model for get structure response
class EssayEvalutionSchema(BaseModel):
    feedback: str = Field(description="Details feedback for generated essay")
    score: int = Field(description="Score for the essay based on evaluation criteria", ge=0, le=10)

In [49]:
structured_model = chat_model.with_structured_output(EssayEvalutionSchema)

In [50]:
class EssayEvalutionState(TypedDict):
    essay_text: str

    # output values
    feedback_language: str
    feedback_clarity_of_thought: str
    feedback_depth_analysis: str
    overall_feedback: str
    evaluation_score: Annotated[list[int], operator.add]
    avg_score: float

In [51]:
# Define graph
graph = StateGraph(EssayEvalutionState)

In [52]:
# Node
def get_language_feedback(state: EssayEvalutionState):
    # Implement logic to evaluate language
    prompt = f"Evaluate the language of the following essay: {state['essay_text']}. Provide feedback on grammar, vocabulary, and overall language quality. and generate score out of 10."
    result = structured_model.invoke(prompt)
    return {'feedback_language': result.feedback, 'evaluation_score': [result.score]} 

def get_clarity_of_thought_feedback(state: EssayEvalutionState):
    # Implement logic to evaluate clarity of thought
    prompt = f"Evaluate the clarity of thought in the following essay: {state['essay_text']}.  and generate score out of 10."
    result = structured_model.invoke(prompt)
    return {'feedback_clarity_of_thought': result.feedback, 'evaluation_score': [result.score]}

def get_depth_of_analysis_feedback(state: EssayEvalutionState):
    # Implement logic to evaluate depth of analysis
    prompt = f"Evaluate the depth of analysis in the following essay: {state['essay_text']}.  and generate score out of 10."
    result = structured_model.invoke(prompt)
    return {'feedback_depth_analysis': result.feedback, 'evaluation_score': [result.score]}

def get_overall_feedback(state: EssayEvalutionState):
    # Implement logic to provide overall feedback
    prompt = f"Provide overall feedback for the following essay: {state['essay_text']}. based on {state['feedback_language']}, {state['feedback_clarity_of_thought']} and {state['feedback_depth_analysis']}"
    result = chat_model.invoke(prompt).content
    avg_score = sum(state['evaluation_score']) / len(state['evaluation_score'])
    return {'overall_feedback': result, 'avg_score': avg_score}
    

In [53]:
# Define nodes in the graph
graph.add_node('language_feedback', get_language_feedback)
graph.add_node('clarity_of_thought_feedback', get_clarity_of_thought_feedback)
graph.add_node('depth_of_analysis_feedback', get_depth_of_analysis_feedback)
graph.add_node('overall_feedback', get_overall_feedback)

In [54]:
# Define edges in the graph
graph.add_edge(START, 'language_feedback')
graph.add_edge(START, 'clarity_of_thought_feedback')
graph.add_edge(START, 'depth_of_analysis_feedback')
graph.add_edge('language_feedback', 'overall_feedback')
graph.add_edge('clarity_of_thought_feedback', 'overall_feedback')
graph.add_edge('depth_of_analysis_feedback', 'overall_feedback')
graph.add_edge('overall_feedback', END)

In [55]:
# Compile
workflow = graph.compile()

In [56]:
essay_upsc_text = f"""
Effect of Artificial Intelligence on the Indian Stock Market

Artificial Intelligence (AI) has emerged as a transformative force in the 21st century, influencing diverse sectors ranging from healthcare to governance. In the financial domain, particularly the stock market, AI is redefining traditional mechanisms of trading, investment analysis, and risk management. In the Indian context, where the stock market plays a crucial role in economic growth and capital formation, the integration of AI presents both significant opportunities and complex challenges.

At its core, AI refers to the ability of machines to simulate human intelligence processes such as learning, reasoning, and decision-making. In the Indian stock market, AI is primarily deployed through algorithmic trading, machine learning models, and sentiment analysis tools. These technologies enable faster data processing, improved prediction accuracy, and enhanced decision-making capabilities.

One of the most visible impacts of AI is in algorithmic trading. AI-powered trading systems can execute large volumes of transactions within milliseconds, based on predefined rules and real-time data analysis. This has increased market efficiency and liquidity while reducing human biases in trading decisions. Additionally, machine learning models are increasingly used to analyze historical data and identify patterns, thereby aiding in stock price prediction. Such predictive analytics empower investors to make informed decisions, especially in a volatile market environment.

Another important application is sentiment analysis. By analyzing news articles, social media trends, and financial reports, AI systems can gauge market sentiment and predict potential movements in stock prices. In a diverse and rapidly evolving market like India, where investor behavior is often influenced by external narratives, this capability holds immense value.

The benefits of AI in the Indian stock market are substantial. It enhances operational efficiency, reduces transaction costs, and improves accuracy in forecasting. Furthermore, AI-driven risk management tools help in identifying potential threats and mitigating losses. For retail investors, AI-enabled platforms provide personalized investment advice, thereby democratizing access to financial markets.

However, the increasing reliance on AI also raises several concerns. Data privacy and security remain critical issues, as AI systems depend on vast amounts of sensitive financial data. There is also the risk of market manipulation through sophisticated algorithms, which may exploit loopholes faster than regulatory mechanisms can respond. Ethical concerns arise regarding the opacity of AI decision-making processes, often referred to as the “black box” problem.

From a regulatory perspective, the challenge lies in balancing innovation with oversight. The Securities and Exchange Board of India (SEBI) has taken steps to regulate algorithmic trading, but the rapid evolution of AI technologies necessitates continuous policy adaptation. Ensuring transparency, accountability, and fairness in AI-driven markets is essential to maintain investor confidence.

Case studies from Indian firms illustrate the growing adoption of AI. Companies like Zerodha use AI for customer insights and trading tools, while financial institutions such as IIFL and Reliance Securities leverage AI for predictive analytics and sentiment analysis. These examples highlight the practical utility of AI while also underscoring the need for robust governance frameworks.

Looking ahead, the role of AI in the Indian stock market is expected to expand further. With advancements in deep learning and big data analytics, AI could revolutionize portfolio management, fraud detection, and regulatory compliance. However, the human element will continue to remain relevant. The future lies in a hybrid model where human expertise complements AI capabilities.

In conclusion, Artificial Intelligence is reshaping the Indian stock market by enhancing efficiency, accuracy, and accessibility. While the benefits are undeniable, it is imperative to address the associated challenges through effective regulation and ethical considerations. A balanced approach that fosters innovation while safeguarding market integrity will be crucial in harnessing the full potential of AI in India’s financial ecosystem.
"""

In [57]:
essay_by_kid = f"""
My Favorite Season

My favorite season is winter because I like cold weather very much. In winter, the air feels nice and I can wear my warm jacket and sweaters which look very cozy. Sometimes it is too cold also but still I enjoy it.

In winter mornings, it is very hard to wake up early because bed feels very warm and comfortable. But when I go outside, I see fog and everything looks white and little mysterious. I like that view a lot. Also, we get to drink hot tea and eat hot food which tastes very good in cold days.

Another reason I like winter is because of holidays. We get some vacations and I can stay at home and watch TV or play games. Sometimes I go outside to play with my friends but only when it is not too much cold, otherwise my hands start paining.

But winter also have some problems. Many people get sick like cold and cough. Even I also get sick sometimes which I don’t like. Also, poor people feel very difficult because they don’t have enough warm clothes.

In conclusion, winter is my favorite season because it feels nice and cozy, even though it has some problems. I enjoy it more than summer because summer is too hot and makes me tired.    
"""

In [60]:
# Define initial state
initial_state = {'essay_text': essay_upsc_text}
result = workflow.invoke(
    initial_state
)

print(result)

{'essay_text': '\nEffect of Artificial Intelligence on the Indian Stock Market\n\nArtificial Intelligence (AI) has emerged as a transformative force in the 21st century, influencing diverse sectors ranging from healthcare to governance. In the financial domain, particularly the stock market, AI is redefining traditional mechanisms of trading, investment analysis, and risk management. In the Indian context, where the stock market plays a crucial role in economic growth and capital formation, the integration of AI presents both significant opportunities and complex challenges.\n\nAt its core, AI refers to the ability of machines to simulate human intelligence processes such as learning, reasoning, and decision-making. In the Indian stock market, AI is primarily deployed through algorithmic trading, machine learning models, and sentiment analysis tools. These technologies enable faster data processing, improved prediction accuracy, and enhanced decision-making capabilities.\n\nOne of the 